In [18]:
import torch
import numpy as np
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import os
from PIL import Image
import torch.nn.functional as F
import seaborn as sns
import matplotlib.pyplot as plt
from torch.cuda.amp import GradScaler, autocast  # Set device for GPU usage
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"CUDA Available: {torch.cuda.is_available()}, Device Name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(torch.__version__)  # E.g., 2.4.0+cu118
print(torch.cuda.is_available())  # Should print True
from torch.utils.data import random_split


Using device: cuda
CUDA Available: True, Device Name: NVIDIA GeForce RTX 4060 Laptop GPU
2.3.1
True


# DATA

In [31]:
genuine_extract_path = './content/Offline Genuine'
forgery_extract_path = './content/Offline Forgeries'
print(f"Genuine files: {os.listdir(genuine_extract_path)}")
print(f"Forgery files: {os.listdir(forgery_extract_path)}")

class SignatureVerificationDataset(Dataset):
    def __init__(self, genuine_dir, forgery_dir, transform=None):
        """
        Args:
            genuine_dir (str): Path to offline_genuine folder (e.g., './data/offline_genuine').
            forgery_dir (str): Path to offline_forgeries folder (e.g., './data/offline_forgeries').
            transform (callable, optional): Transformations to apply to images.
        """
        self.genuine_dir = genuine_dir
        self.forgery_dir = forgery_dir
        self.transform = transform
        self.pairs = []
        self.labels = []

        # Dictionary to store signatures by author
        genuine_by_author = {}
        forgery_by_author = {}

        # Load genuine signatures
        for filename in os.listdir(genuine_dir):
            if filename.endswith(('.png', '.jpg', '.jpeg','.PNG')):
                try:
                    # Parse filename, e.g., '001_02.png' -> author '001'
                    author_id = filename.split('_')[0]
                    img_path = os.path.join(genuine_dir, filename)
                    if author_id not in genuine_by_author:
                        genuine_by_author[author_id] = []
                    genuine_by_author[author_id].append(img_path)
                except (ValueError, IndexError):
                    print(f"Skipping invalid genuine filename: {filename}")
                    continue

        # Load forged signatures
        for filename in os.listdir(forgery_dir):
            if filename.endswith(('.png', '.jpg', '.jpeg')):
                try:
                    # Parse filename, e.g., '0119001_01.png' -> author '001'
                    parts = filename.split('_')
                    author_id = parts[0][-3:]  # Extract last 3 digits (e.g., '001' from '0119001')
                    img_path = os.path.join(forgery_dir, filename)
                    if author_id not in forgery_by_author:
                        forgery_by_author[author_id] = []
                    forgery_by_author[author_id].append(img_path)
                except (ValueError, IndexError):
                    print(f"Skipping invalid forgery filename: {filename}")
                    continue

        print(f"Genuine authors: {len(genuine_by_author)}, {genuine_by_author.keys()}")
        print(f"Forgery authors: {len(forgery_by_author)}, {forgery_by_author.keys()}")
        for author_id, sigs in genuine_by_author.items():
            print(f"Author {author_id}: {len(sigs)} genuine signatures")
        for author_id, sigs in forgery_by_author.items():
            print(f"Author {author_id}: {len(sigs)} forged signatures")

        # Create pairs for each author
        for author_id in genuine_by_author:
            genuine_sigs = genuine_by_author.get(author_id, [])
            forged_sigs = forgery_by_author.get(author_id, [])
            if not (forged_sigs or genuine_sigs):
                print(f"Warning: No data found for author {author_id}")

            # Genuine-Genuine pairs (label = 1)
            for i in range(len(genuine_sigs)):
                for j in range(i + 1, len(genuine_sigs)):  # Pair different genuine samples
                    self.pairs.append((genuine_sigs[i], genuine_sigs[j]))
                    self.labels.append(1)

            # Genuine-Forgery pairs (label = 0)
            for genuine_sig in genuine_sigs:
                for forged_sig in forged_sigs:
                    self.pairs.append((genuine_sig, forged_sig))
                    self.labels.append(0)

        print(f"Training Total pairs: {len(self.pairs)}, Total labels: {len(self.labels)}")
        count=0
        for pair in self.pairs:
            print(f"all: {pair} {self.labels[count]}")
            count+=1
        print(f"Training Positive pairs (label=1): {sum(1 for label in self.labels if label == 1)}")
        print(f"Training Negative pairs (label=0): {sum(1 for label in self.labels if label == 0)}")

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        # Get pair of image paths and label
        img1_path, img2_path = self.pairs[idx]
        label = self.labels[idx]

        # Load images
        img1 = Image.open(img1_path).convert('L')  # Grayscale, like FashionMNIST
        img2 = Image.open(img2_path).convert('L')

        # Apply transforms
        if self.transform:
            img1 = self.transform(img1)
            img2 = self.transform(img2)


        return img1, img2, label

# Example usage
transform = transforms.Compose([
    transforms.Resize((64, 64)),  # Resize to match FashionMNIST if needed
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # Standard normalization
])

train_dataset = SignatureVerificationDataset(
    genuine_dir=genuine_extract_path,
    forgery_dir=forgery_extract_path,
    transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True, num_workers=0, pin_memory=True)


Genuine files: ['.ipynb_checkpoints', '001_01.PNG', '001_02.PNG', '001_03.PNG', '001_04.PNG', '001_05.PNG', '001_06.PNG', '001_07.PNG', '001_08.PNG', '001_09.PNG', '001_10.PNG', '001_11.PNG', '001_12.PNG', '001_13.PNG', '001_14.PNG', '001_15.PNG', '001_16.PNG', '001_17.PNG', '001_18.PNG', '001_19.PNG', '001_20.PNG', '001_21.PNG', '001_22.PNG', '001_23.PNG', '001_24.PNG', '002_01.PNG', '002_02.PNG', '002_03.PNG', '002_04.PNG', '002_05.PNG', '002_06.PNG', '002_07.PNG', '002_08.PNG', '002_09.PNG', '002_10.PNG', '002_11.PNG', '002_12.PNG', '002_13.PNG', '002_14.PNG', '002_15.PNG', '002_16.PNG', '002_17.PNG', '002_18.PNG', '002_19.PNG', '002_20.PNG', '002_21.PNG', '002_22.PNG', '002_23.PNG', '002_24.PNG', '003_01.PNG', '003_02.PNG', '003_03.PNG', '003_04.PNG', '003_05.PNG', '003_06.PNG', '003_07.PNG', '003_08.PNG', '003_09.PNG', '003_10.PNG', '003_11.PNG', '003_12.PNG', '003_13.PNG', '003_14.PNG', '003_15.PNG', '003_16.PNG', '003_17.PNG', '003_18.PNG', '003_19.PNG', '003_20.PNG', '003_21.PN

In [32]:
reference_extract_path='./content/Reference(646)'
ques_extract_path = './content/Questioned(1287)'
print(f"Reference files: {os.listdir(reference_extract_path)}")
print(f"Questioned files: {os.listdir(ques_extract_path)}")

class SignatureVerificationTestDataset(Dataset):
    def __init__(self, reference_dir, ques_dir, transform=None):
        """
        Args:
            reference_dir (str): Path to offline_genuine folder (e.g., './data/offline_genuine').
            ques_dir (str): Path to offline_forgeries folder (e.g., './data/offline_forgeries').
            transform (callable, optional): Transformations to apply to images.
        """
        self.reference_dir = reference_dir
        self.ques_dir = ques_dir
        self.transform = transform
        self.pairs = []
        self.labels = []

        # Dictionary to store signatures by author
        reference_by_author = {}
        ques_genuine_by_author = {}
        ques_forgery_by_author = {}

        # Load reference signatures
        for author_id in os.listdir(reference_dir):
            for filename in os.listdir(os.path.join(reference_dir,author_id)):
                if filename.endswith(('.png', '.jpg', '.jpeg','.PNG')):
                    try:
                        # Parse filename, e.g., '001_02.png' -> author '001'
                        img_path = os.path.join(reference_dir,author_id, filename)
                        if author_id not in reference_by_author:
                            reference_by_author[author_id] = []
                        reference_by_author[author_id].append(img_path)
                    except (ValueError, IndexError):
                        print(f"Skipping invalid genuine filename: {filename}")
                        continue

        print("ref",reference_by_author)
        # Load questioned genuine signatures
        for author_id in os.listdir(ques_dir):
            for filename in os.listdir(os.path.join(ques_dir,author_id)):
                if filename.endswith(('.png', '.jpg', '.jpeg')):
                    try:
                        # Parse filename, e.g., '001_02.png' -> author '001'
                        img_path = os.path.join(ques_dir,author_id, filename)
                        if author_id not in ques_genuine_by_author:
                            ques_genuine_by_author[author_id] = []
                        ques_genuine_by_author[author_id].append(img_path)
                    except (ValueError, IndexError):
                        print(f"Skipping invalid genuine filename: {filename}")
                        continue
        print("ques",ques_genuine_by_author)

        # Load questioned forgery signatures
        for author_id in os.listdir(ques_dir):
            for filename in os.listdir(os.path.join(ques_dir,author_id)):
                if filename.endswith(('.PNG')):
                    try:
                        # Parse filename, e.g., '001_02.png' -> author '001'
                        img_path = os.path.join(ques_dir,author_id, filename)
                        if author_id not in ques_forgery_by_author:
                            ques_forgery_by_author[author_id] = []
                        ques_forgery_by_author[author_id].append(img_path)
                    except (ValueError, IndexError):
                        print(f"Skipping invalid genuine filename: {filename}")
                        continue
        print("ques",ques_forgery_by_author)

        # Create pairs for each author
        for author_id in reference_by_author:
            reference_sigs = reference_by_author.get(author_id, [])
            ques_genuine_sigs= ques_genuine_by_author.get(author_id,[])
            ques_forged_sigs = ques_forgery_by_author.get(author_id, [])
            if not (reference_sigs or ques_forgery_by_author or ques_genuine_by_author):
                print(f"Warning: No data found for author {author_id}")

            # Reference-Genuine pairs (label = 1)
            for i in range(len(reference_sigs)):
                for j in range(len(ques_genuine_sigs)):  # Pair different genuine samples
                    self.pairs.append((reference_sigs[i], ques_genuine_sigs[j]))
                    self.labels.append(1)

            # Reference-Forgery pairs (label = 0)
            for i in range(len(reference_sigs)):
                for j in range(len(ques_forged_sigs)):  # Pair different genuine samples
                    self.pairs.append((reference_sigs[i], ques_forged_sigs[j]))
                    self.labels.append(0)

        print(f"Training Total pairs: {len(self.pairs)}, Total labels: {len(self.labels)}")
        count=0
        for pair in self.pairs:
            print(f"all: {pair} {self.labels[count]}")
            count+=1
        print(f"Training Positive pairs (label=1): {sum(1 for label in self.labels if label == 1)}")
        print(f"Training Negative pairs (label=0): {sum(1 for label in self.labels if label == 0)}")

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        # Get pair of image paths and label
        img1_path, img2_path = self.pairs[idx]
        label = self.labels[idx]

        # Load images
        img1 = Image.open(img1_path).convert('L')  # Grayscale, like FashionMNIST
        img2 = Image.open(img2_path).convert('L')

        # Apply transforms
        if self.transform:
            img1 = self.transform(img1)
            img2 = self.transform(img2)

        label = torch.tensor(label, dtype=torch.float32)

        return img1, img2, label

# Example usage
transform = transforms.Compose([
    transforms.Resize((64, 64)),  # Resize to match FashionMNIST if needed
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # Standard normalization
])

testdataset = SignatureVerificationTestDataset(
    reference_extract_path,
    ques_extract_path,
    transform=transform
)

test_size = int(0.8 * len(testdataset))
val_size = len(testdataset) - test_size

test_dataset, val_dataset = random_split(testdataset,[test_size,val_size])

val_loader = DataLoader(val_dataset, batch_size=256, shuffle=True, num_workers=0, pin_memory=True)

testdataloader = DataLoader(
    test_dataset,
    batch_size=256,  # Reduced for laptop GPU
    shuffle=True,
    num_workers=0,  # Adjusted for performance (set to 0 for debugging)
    pin_memory=True
)

Reference files: ['013', '017', '018', '019', '020', '021', '022', '023', '024', '025', '026', '027', '028', '029', '030', '031', '032', '033', '034', '035', '036', '037', '038', '039', '040', '041', '042', '043', '044', '045', '046', '047', '048', '049', '050', '051', '052', '053', '054', '055', '056', '057', '058', '059', '060', '061', '062', '063', '064', '065', '066', '067', '068', '069']
Questioned files: ['013', '017', '018', '019', '020', '021', '022', '023', '024', '025', '026', '027', '028', '029', '030', '031', '032', '033', '034', '035', '036', '037', '038', '039', '040', '041', '042', '043', '044', '045', '046', '047', '048', '049', '050', '051', '052', '053', '054', '055', '056', '057', '058', '059', '060', '061', '062', '063', '064', '065', '066', '067', '068', '069']
ref {'013': ['./content/Reference(646)\\013\\13_013.PNG', './content/Reference(646)\\013\\14_013.PNG', './content/Reference(646)\\013\\15_013.PNG', './content/Reference(646)\\013\\16_013.PNG', './content/Ref

In [33]:
class SiameseCNN(nn.Module):
    def __init__(self):
        super(SiameseCNN, self).__init__()
        self.cnn =nn.Sequential(
        nn.Conv2d(1, 32, 3, padding=1),
        nn.BatchNorm2d(32),
        nn.ReLU(),
        nn.MaxPool2d(2),
    
        nn.Conv2d(32, 64, 3, padding=1),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.MaxPool2d(2),
    
        nn.Conv2d(64, 128, 3, padding=1),
        nn.BatchNorm2d(128),
        nn.ReLU(),
        nn.MaxPool2d(2),
    
        nn.Flatten(),
        nn.Linear(128 * 8 * 8, 256),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(256, 64)
)

    def forward(self, x1, x2):
        embedding1 = self.cnn(x1)
        embedding2 = self.cnn(x2)
        return embedding1, embedding2

In [34]:
class ContrastiveLoss(nn.Module):
    def __init__(self, margin=1.0):
        super(ContrastiveLoss, self).__init__()
        self.margin = margin

    def forward(self, emb1, emb2, label):
        cosine_sim = nn.functional.cosine_similarity(emb1, emb2)
        loss = label * (1 - cosine_sim) + (1 - label) * torch.clamp(cosine_sim - self.margin, min=0)
        return loss.mean()
    
model = SiameseCNN()


In [35]:
# Define margins and learning rates to try
margins = [0.8]
learning_rates = [0.01]
num_epochs = 10
threshold = 0.9
# Create directory to save models
os.makedirs('second_saved_models', exist_ok=True)

for margin in margins:
    for lr in learning_rates:
        print(f"\n--- Training with margin={margin}, learning_rate={lr} ---\n")
        
        # Initialize model, loss, optimizer
        model = SiameseCNN().to(device)
        criterion = ContrastiveLoss(margin=margin)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        epoch_losses = []

        # Training loop
        for epoch in range(num_epochs):
            model.train()
            batch_losses = []
            for img1, img2, labels in train_loader:
                img1, img2, labels = img1.to(device), img2.to(device), labels.to(device)

                optimizer.zero_grad()
                emb1, emb2 = model(img1, img2)
                loss = criterion(emb1, emb2, labels)
                loss.backward()
                optimizer.step()

                batch_losses.append(loss.item())

            avg_loss = sum(batch_losses) / len(batch_losses)
            epoch_losses.append(avg_loss)
            print(f"Epoch {epoch+1}/{num_epochs} | Margin: {margin} | LR: {lr} | Avg Loss: {avg_loss:.4f}")

        model.eval()
        correct = 0
        total = 0
        cosine_sim_all = []
        labels_all = []
        with torch.no_grad():
            for img1, img2, labels in val_loader:
                img1 = img1.to(device)
                img2 = img2.to(device)
                labels = labels.to(device)
                try:
                    with autocast():
                        emb1, emb2 = model(img1, img2)
                        emb1 = F.normalize(emb1, p=2, dim=1)  # Normalize embeddings
                        emb2 = F.normalize(emb2, p=2, dim=1)
                        cosine_sim = F.cosine_similarity(emb1, emb2)
                    cosine_sim_all.append(cosine_sim)
                    labels_all.append(labels)
                    predictions = (cosine_sim > threshold).float()
                    correct += (predictions == labels).sum().item()
                    total += labels.size(0)
                except RuntimeError as e:
                    print(f"Error in evaluation: {e}")
                    if "out of memory" in str(e):
                        torch.cuda.empty_cache()
                        continue
    
        cosine_sim_all = torch.cat(cosine_sim_all)
        labels_all = torch.cat(labels_all)
    
         #Plot histogram
        indices = np.random.permutation(total)[:1000]
        cosine_sim_np = cosine_sim_all[indices].cpu().numpy()
        labels_np = labels_all[indices].cpu().numpy()
        cosine_sim_genuine = cosine_sim_np[labels_np == 1].tolist()
        cosine_sim_forged = cosine_sim_np[labels_np == 0].tolist()
    
    
        plt.figure(figsize=(8, 6))
        sns.histplot(cosine_sim_genuine, color='blue', label='Genuine Pairs', kde=True, stat='density')
        sns.histplot(cosine_sim_forged, color='red', label='Forged Pairs', kde=True, stat='density')
        plt.axvline(x=threshold, color='green', linestyle='--', label=f'Threshold ({threshold})')
        plt.xlabel('Cosine Similarity')
        plt.ylabel('Density')
        plt.title(f'Cosine Similarity Distribution for Genuine and Forged Pairs {threshold}_{margin}_{lr}')
        plt.legend()
        plt.grid(True)
        # plt.savefig(f"cosine_similarity_distribution_threshold_{threshold}_{margin}_{lr}.png")
        plt.show()
        
        val_accuracy = correct / total if total > 0 else 0.0
        print(f"Validation accuracy={val_accuracy}")
        # Save model
        model_path = f"second_saved_models/siamese_margin{margin}_lr{lr}.pt"
        torch.save(model.state_dict(), model_path)
        print(f"Model saved to {model_path}")

        # Plot loss curve
        plt.figure(figsize=(8, 6))
        plt.plot(range(1, num_epochs + 1), epoch_losses, marker='o', label=f"m={margin}, lr={lr}")
        plt.xlabel('Epoch')
        plt.ylabel('Contrastive Loss')
        plt.title(f'Loss Curve (margin={margin}, lr={lr})')
        plt.legend()
        plt.grid(True)
        plt.show
        # plt.savefig(f'second_saved_models/loss_margin_LENET{margin}_lr{lr}.png')




--- Training with margin=0.8, learning_rate=0.01 ---

Epoch 1/10 | Margin: 0.8 | LR: 0.01 | Avg Loss: 0.1097
Epoch 2/10 | Margin: 0.8 | LR: 0.01 | Avg Loss: 0.1019
Epoch 3/10 | Margin: 0.8 | LR: 0.01 | Avg Loss: 0.0943
Epoch 4/10 | Margin: 0.8 | LR: 0.01 | Avg Loss: 0.0762
Epoch 5/10 | Margin: 0.8 | LR: 0.01 | Avg Loss: 0.0532
Epoch 6/10 | Margin: 0.8 | LR: 0.01 | Avg Loss: 0.0315
Epoch 7/10 | Margin: 0.8 | LR: 0.01 | Avg Loss: 0.0104


KeyboardInterrupt: 

In [90]:
# PATH = './model/mymodellenet.pth'
# torch.save(model.state_dict(),PATH)

In [91]:
# # Load the model
# model = SiameseCNN().to(device)
# try:
#     model.load_state_dict(torch.load(PATH, map_location=device))
# except RuntimeError as e:
#     print(f"Error loading model: {e}")


In [1]:
plt.close('all')  # Closes all figures
torch.cuda.empty_cache()
model = SiameseCNN().to(device)
def evaluate_accuracy(model, testdataloader, margin, lr, threshold=0.5,device="cuda" if torch.cuda.is_available() else "cpu"):
    model.eval()
    correct = 0
    total = 0
    cosine_sim_all = []
    labels_all = []
    with torch.no_grad():
        for img1, img2, labels in testdataloader:
            img1 = img1.to(device)
            img2 = img2.to(device)
            labels = labels.to(device)
            try:
                with autocast():
                    emb1, emb2 = model(img1, img2)
                    emb1 = F.normalize(emb1, p=2, dim=1)  # Normalize embeddings
                    emb2 = F.normalize(emb2, p=2, dim=1)
                    cosine_sim = F.cosine_similarity(emb1, emb2)
                cosine_sim_all.append(cosine_sim)
                labels_all.append(labels)
                predictions = (cosine_sim > threshold).float()
                correct += (predictions == labels).sum().item()
                total += labels.size(0)
            except RuntimeError as e:
                print(f"Error in evaluation: {e}")
                if "out of memory" in str(e):
                    torch.cuda.empty_cache()
                    continue

    cosine_sim_all = torch.cat(cosine_sim_all)
    labels_all = torch.cat(labels_all)

     #Plot histogram
    indices = np.random.permutation(total)[:1000]
    cosine_sim_np = cosine_sim_all[indices].cpu().numpy()
    labels_np = labels_all[indices].cpu().numpy()
    cosine_sim_genuine = cosine_sim_np[labels_np == 1].tolist()
    cosine_sim_forged = cosine_sim_np[labels_np == 0].tolist()


    plt.figure(figsize=(8, 6))
    sns.histplot(cosine_sim_genuine, color='blue', label='Genuine Pairs', kde=True, stat='density')
    sns.histplot(cosine_sim_forged, color='red', label='Forged Pairs', kde=True, stat='density')
    plt.axvline(x=threshold, color='green', linestyle='--', label=f'Threshold ({threshold})')
    plt.xlabel('Cosine Similarity')
    plt.ylabel('Density')
    plt.title(f'Cosine Similarity Distribution for Genuine and Forged Pairs {threshold}_{margin}_{lr}')
    plt.legend()
    plt.grid(True)
    plt.savefig(f"cosine_similarity_distribution_threshold_{threshold}_{margin}_{lr}.png")
    plt.show()
    
    accuracy = correct / total if total > 0 else 0.0
    return accuracy

# Test multiple thresholds
thresholds = [0.7]
# Load the model

for filename in os.listdir('./saved_models'):
    if filename.endswith(('.pt')):
        PATH=os.path.join('./saved_models',filename)
        lr=filename.split("_")[2]
        constraint=filename.split("_")[1][:7]
        if (constraint == 'margin0'):
            margin=filename.split("_")[1]
            if margin=="margin0.5" and lr=="lr0.0005.pt":
                print(margin,lr)
                model.load_state_dict(torch.load(PATH, map_location=device))
                for thresh in thresholds:
                    accuracy = evaluate_accuracy(model, testdataloader,margin,lr,threshold=thresh)
                    print(f"Threshold: {thresh}, Accuracy: {accuracy:.4f}")
   

NameError: name 'plt' is not defined